# Task 2: Build Time Series Forecasting Models
## ARIMA/SARIMA and LSTM Comparison for Tesla Stock Price Prediction

**Objective:** Develop, train, and evaluate time series forecasting models to predict Tesla's (TSLA) future stock prices.

This notebook implements:
1. **Classical Statistical Model**: ARIMA/SARIMA using pmdarima
2. **Deep Learning Model**: LSTM neural network using TensorFlow/Keras
3. **Comparative Analysis**: Performance metrics and model selection rationale

**Forecasting Approach**: Predict adjusted close prices
**Train-Test Split**: Chronological (2015-2024 training, 2025-2026 testing)
**Assets**: Focus on TSLA with references to BND and SPY

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Time series and statistical modeling
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
import pmdarima as pm

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set plotting style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Section 1: Data Preparation

We will load the processed TSLA data and perform chronological train-test split.

In [ ]:
# Load TSLA data from Task 1
import os
import yfinance as yf

print("Loading TSLA data...")

# Download data using yfinance (same as Task 1)
START_DATE = "2015-01-01"
END_DATE = "2026-06-30"

# Try to load from processed data if available
data_path = os.path.join(os.getcwd(), 'data', 'processed', 'TSLA_processed.csv')

if os.path.exists(data_path):
    print(f"Loading from: {data_path}")
    tsla_data = pd.read_csv(data_path, index_col=0, parse_dates=True)
    print("✓ Data loaded from processed file")
else:
    print("Downloading from YFinance...")
    tsla_data = yf.download('TSLA', start=START_DATE, end=END_DATE, progress=False)
    print("✓ Data downloaded from YFinance")

# Display data info
print(f"\nTSLA Data Shape: {tsla_data.shape}")
print(f"Date Range: {tsla_data.index.min()} to {tsla_data.index.max()}")
print(f"Total trading days: {len(tsla_data)}")
print(f"\nFirst few rows:\n{tsla_data.head()}")
print(f"\nColumn names: {list(tsla_data.columns)}")

In [ ]:
# Chronological Train-Test Split
print("=" * 80)
print("CHRONOLOGICAL TRAIN-TEST SPLIT")
print("=" * 80)

# Split date: End of 2024
split_date = pd.Timestamp('2024-12-31')

# Training data: 2015-2024
train_data = tsla_data[tsla_data.index <= split_date]

# Testing data: 2025-2026
test_data = tsla_data[tsla_data.index > split_date]

print(f"\nTraining Set:")
print(f"  Date Range: {train_data.index.min()} to {train_data.index.max()}")
print(f"  Number of records: {len(train_data)}")
print(f"  Percentage: {len(train_data) / len(tsla_data) * 100:.1f}%")

print(f"\nTest Set:")
print(f"  Date Range: {test_data.index.min()} to {test_data.index.max()}")
print(f"  Number of records: {len(test_data)}")
print(f"  Percentage: {len(test_data) / len(tsla_data) * 100:.1f}%")

print(f"\nTotal records: {len(tsla_data)}")

# Use Adj Close for modeling
price_column = 'Adj Close' if 'Adj Close' in tsla_data.columns else 'Close'

train_prices = train_data[[price_column]].copy()
test_prices = test_data[[price_column]].copy()

print(f"\nUsing '{price_column}' column for forecasting")
print(f"Training price range: ${train_prices.min().values[0]:.2f} - ${train_prices.max().values[0]:.2f}")
print(f"Test price range: ${test_prices.min().values[0]:.2f} - ${test_prices.max().values[0]:.2f}")

# Visualize the split
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(train_data.index, train_prices, linewidth=2, label='Training Data', color='blue', alpha=0.8)
ax.plot(test_data.index, test_prices, linewidth=2, label='Test Data', color='red', alpha=0.8)
ax.axvline(x=split_date, color='black', linestyle='--', linewidth=2, label=f'Split Date ({split_date.date()})')

ax.set_xlabel('Date', fontsize=11, fontweight='bold')
ax.set_ylabel('Price ($)', fontsize=11, fontweight='bold')
ax.set_title('TSLA: Chronological Train-Test Split', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('split_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Train-test split completed successfully")

## Section 2: ARIMA/SARIMA Model

We will use auto_arima to find optimal parameters and train the model.

In [ ]:
# ACF and PACF plots to understand patterns
print("=" * 80)
print("ACF/PACF ANALYSIS FOR PARAMETER SELECTION")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Original series
axes[0, 0].plot(train_prices, linewidth=1)
axes[0, 0].set_title('TSLA Original Price Series', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].grid(True, alpha=0.3)

# First differencing
diff_series = train_prices.diff().dropna()
axes[0, 1].plot(diff_series, linewidth=1, color='orange')
axes[0, 1].set_title('First Differenced Series (d=1)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Price Change ($)')
axes[0, 1].grid(True, alpha=0.3)

# ACF of original
plot_acf(train_prices.dropna(), lags=40, ax=axes[1, 0])
axes[1, 0].set_title('ACF of Original Series', fontsize=11, fontweight='bold')

# ACF of differenced
plot_acf(diff_series, lags=40, ax=axes[1, 1])
axes[1, 1].set_title('ACF of Differenced Series', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('acf_pacf_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nACF/PACF plots created successfully")

In [ ]:
# Find optimal ARIMA parameters using auto_arima
print("\n" + "=" * 80)
print("AUTO_ARIMA: FINDING OPTIMAL PARAMETERS")
print("=" * 80)
print("This may take 1-2 minutes...\n")

# Extract price series for modeling
train_series = train_prices.values.flatten()

# auto_arima with seasonal component (SARIMA)
try:
    print("Running auto_arima with seasonal parameters...")
    auto_arima_result = pm.auto_arima(
        train_series,
        start_p=0, max_p=5,
        start_d=0, max_d=2,
        start_q=0, max_q=5,
        seasonal=True,
        m=252,  # 252 trading days per year
        start_P=0, max_P=2,
        start_D=0, max_D=1,
        start_Q=0, max_Q=2,
        trace=False,
        error_action='ignore',
        suppress_warnings=True,
        stepwise=True
    )
    
    print("✓ Auto_arima completed successfully!")
    print(f"\nOptimal SARIMA Parameters: {auto_arima_result.order} x {auto_arima_result.seasonal_order}")
    print(f"  (p, d, q): {auto_arima_result.order}")
    print(f"  (P, D, Q, m): {auto_arima_result.seasonal_order}")
    print(f"\nAIC: {auto_arima_result.aic:.2f}")
    print(f"BIC: {auto_arima_result.bic:.2f}")
    
    # Store best parameters
    best_order = auto_arima_result.order
    best_seasonal_order = auto_arima_result.seasonal_order
    
    print("\n" + auto_arima_result.summary().as_text()[:500] + "...")
    
except Exception as e:
    print(f"Note: Seasonal ARIMA failed ({str(e)[:50]}), falling back to ARIMA...")
    # Fallback to ARIMA without seasonal component
    auto_arima_result = pm.auto_arima(
        train_series,
        start_p=0, max_p=5,
        start_d=0, max_d=2,
        start_q=0, max_q=5,
        seasonal=False,
        trace=False,
        error_action='ignore',
        suppress_warnings=True,
        stepwise=True
    )
    
    print("✓ Auto_arima (non-seasonal) completed!")
    print(f"\nOptimal ARIMA Parameters: {auto_arima_result.order}")
    print(f"  (p, d, q): {auto_arima_result.order}")
    print(f"AIC: {auto_arima_result.aic:.2f}")
    
    best_order = auto_arima_result.order
    best_seasonal_order = None

In [ ]:
# Train ARIMA/SARIMA model on full training data
print("\n" + "=" * 80)
print("TRAINING ARIMA/SARIMA MODEL")
print("=" * 80)

if best_seasonal_order is not None:
    print(f"\nTraining SARIMA{best_order}{best_seasonal_order}...")
    try:
        arima_model = pm.ARIMA(order=best_order, seasonal_order=best_seasonal_order)
        arima_model.fit(train_series)
        model_type = "SARIMA"
    except:
        print("SARIMA fitting failed, using ARIMA...")
        arima_model = pm.ARIMA(order=best_order)
        arima_model.fit(train_series)
        model_type = "ARIMA"
else:
    print(f"\nTraining ARIMA{best_order}...")
    arima_model = pm.ARIMA(order=best_order)
    arima_model.fit(train_series)
    model_type = "ARIMA"

print(f"✓ {model_type} model trained successfully!")

# Make predictions on test set
test_steps = len(test_series)
arima_predictions, arima_conf_int = arima_model.predict(
    n_periods=test_steps,
    return_conf_int=True,
    alpha=0.05
)

print(f"\nGenerated {len(arima_predictions)} predictions for test period")
print(f"First 5 predictions: {arima_predictions[:5]}")
print(f"Last 5 predictions: {arima_predictions[-5:]}")

# Store test series values
test_series_values = test_prices.values.flatten()

# Create a dataframe for easier comparison
arima_results_df = pd.DataFrame({
    'Date': test_prices.index,
    'Actual': test_series_values,
    'Predicted': arima_predictions,
    'Lower_CI': arima_conf_int[:, 0],
    'Upper_CI': arima_conf_int[:, 1]
})

print(f"\nARIMA Results Summary:")
print(arima_results_df.head(10))

## Section 3: LSTM Deep Learning Model

We will prepare sequence data and build an LSTM neural network for price prediction.

In [ ]:
# Prepare data for LSTM
print("\n" + "=" * 80)
print("PREPARING DATA FOR LSTM MODEL")
print("=" * 80)

# Use all available data for LSTM training (larger dataset for neural networks)
all_prices = tsla_data[[price_column]].values.flatten()

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(all_prices.reshape(-1, 1)).flatten()

# Create sequences
sequence_length = 60  # Use 60 days to predict next day

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_prices, sequence_length)

print(f"Created {len(X)} sequences with length {sequence_length}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split into train and test
# Find the split index in the scaled data
split_index = len(train_prices)

X_train = X[:split_index - sequence_length]
y_train = y[:split_index - sequence_length]

X_test = X[split_index - sequence_length:]
y_test = y[split_index - sequence_length:]

print(f"\nTrain set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")

print(f"\nTest set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")

print("\n✓ Data preparation for LSTM completed")

In [ ]:
# Build LSTM Model Architecture
print("\n" + "=" * 80)
print("BUILDING LSTM MODEL ARCHITECTURE")
print("=" * 80)

lstm_model = Sequential([
    # Input layer (sequence_length, 1)
    layers.LSTM(50, activation='relu', return_sequences=True, input_shape=(sequence_length, 1)),
    layers.Dropout(0.2),
    
    # Second LSTM layer
    layers.LSTM(50, activation='relu', return_sequences=True),
    layers.Dropout(0.2),
    
    # Third LSTM layer
    layers.LSTM(25, activation='relu'),
    layers.Dropout(0.2),
    
    # Dense layers
    layers.Dense(25, activation='relu'),
    layers.Dense(1)  # Output layer
])

# Compile the model
lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

print("✓ Model Architecture Built Successfully!")
print(f"\nModel Summary:")
lstm_model.summary()

# Store model architecture description
model_config = f"""
LSTM Model Architecture:
- Input Layer: Sequence of {sequence_length} days
- LSTM Layer 1: 50 units, ReLU activation, return_sequences=True
- Dropout: 0.2
- LSTM Layer 2: 50 units, ReLU activation, return_sequences=True
- Dropout: 0.2
- LSTM Layer 3: 25 units, ReLU activation
- Dropout: 0.2
- Dense Layer: 25 units, ReLU activation
- Output Layer: 1 unit (price prediction)
- Optimizer: Adam (lr=0.001)
- Loss: Mean Squared Error
- Metrics: Mean Absolute Error
"""

print("\n" + model_config)

In [ ]:
# Train LSTM Model
print("\n" + "=" * 80)
print("TRAINING LSTM MODEL")
print("=" * 80)
print("This may take 2-3 minutes...\n")

# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train the model
history = lstm_model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print("✓ LSTM Model Training Completed!")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
axes[0].plot(history.history['loss'], linewidth=2, label='Training Loss')
axes[0].plot(history.history['val_loss'], linewidth=2, label='Validation Loss')
axes[0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Loss (MSE)', fontsize=11, fontweight='bold')
axes[0].set_title('LSTM Model: Training and Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], linewidth=2, label='Training MAE')
axes[1].plot(history.history['val_mae'], linewidth=2, label='Validation MAE')
axes[1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1].set_ylabel('MAE ($)', fontsize=11, fontweight='bold')
axes[1].set_title('LSTM Model: Training and Validation MAE', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTraining History:")
print(f"  Final Training Loss: {history.history['loss'][-1]:.6f}")
print(f"  Final Validation Loss: {history.history['val_loss'][-1]:.6f}")
print(f"  Epochs trained: {len(history.history['loss'])}")

In [ ]:
# Make predictions with LSTM model
print("\n" + "=" * 80)
print("MAKING LSTM PREDICTIONS")
print("=" * 80)

# Predictions on test set
lstm_predictions_scaled = lstm_model.predict(X_test, verbose=0)

# Inverse transform to get actual prices
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).flatten()

# Get actual test prices
# Need to align with the sequences used in X_test
actual_test_prices = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

print(f"✓ Generated {len(lstm_predictions)} LSTM predictions")
print(f"First 5 predictions: {lstm_predictions[:5]}")
print(f"Last 5 predictions: {lstm_predictions[-5:]}")

# Create LSTM results dataframe
# The test dates need to be aligned with the sequences
test_dates_lstm = test_prices.index[sequence_length:]

lstm_results_df = pd.DataFrame({
    'Date': test_dates_lstm,
    'Actual': actual_test_prices,
    'Predicted': lstm_predictions
})

print(f"\nLSTM Results Summary:")
print(lstm_results_df.head(10))

## Section 4: Model Evaluation and Comparison

We will calculate performance metrics and compare the ARIMA and LSTM models.

In [ ]:
# Calculate Performance Metrics
print("=" * 80)
print("MODEL PERFORMANCE METRICS")
print("=" * 80)

def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE, and MAPE"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Additional metrics
    mpe = np.mean((y_true - y_pred) / y_true * 100)  # Mean Percentage Error
    
    # Direction accuracy (did the model predict up/down correctly?)
    actual_direction = np.diff(y_true) > 0
    pred_direction = np.diff(y_pred) > 0
    direction_accuracy = np.sum(actual_direction == pred_direction) / len(actual_direction) * 100
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'MPE': mpe,
        'Direction_Accuracy': direction_accuracy
    }

# For ARIMA, we use the full test set
arima_metrics = calculate_metrics(test_series_values, arima_predictions)

# For LSTM, we need to use aligned test data (same length)
# Align LSTM predictions with ARIMA for fair comparison
# Use the common overlap period
lstm_test_actual = lstm_results_df['Actual'].values
lstm_test_pred = lstm_results_df['Predicted'].values
lstm_metrics = calculate_metrics(lstm_test_actual, lstm_test_pred)

# Display metrics
print("\nARIMA Model Metrics:")
print("-" * 40)
for metric, value in arima_metrics.items():
    if metric == 'Direction_Accuracy':
        print(f"  {metric:20s}: {value:>10.2f}%")
    elif metric == 'MAPE':
        print(f"  {metric:20s}: {value:>10.2f}%")
    else:
        print(f"  {metric:20s}: ${value:>10.2f}")

print("\nLSTM Model Metrics:")
print("-" * 40)
for metric, value in lstm_metrics.items():
    if metric == 'Direction_Accuracy':
        print(f"  {metric:20s}: {value:>10.2f}%")
    elif metric == 'MAPE':
        print(f"  {metric:20s}: {value:>10.2f}%")
    else:
        print(f"  {metric:20s}: ${value:>10.2f}")

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Metric': ['MAE ($)', 'RMSE ($)', 'MAPE (%)', 'MPE (%)', 'Direction Accuracy (%)'],
    'ARIMA': [
        arima_metrics['MAE'],
        arima_metrics['RMSE'],
        arima_metrics['MAPE'],
        arima_metrics['MPE'],
        arima_metrics['Direction_Accuracy']
    ],
    'LSTM': [
        lstm_metrics['MAE'],
        lstm_metrics['RMSE'],
        lstm_metrics['MAPE'],
        lstm_metrics['MPE'],
        lstm_metrics['Direction_Accuracy']
    ]
})

# Calculate which is better
comparison_df['Better Model'] = comparison_df.apply(
    lambda row: 'LSTM' if (row['Metric'] in ['MAE ($)', 'RMSE ($)', 'MAPE (%)', 'MPE (%)'] and row['LSTM'] < row['ARIMA']) 
                or (row['Metric'] == 'Direction Accuracy (%)' and row['LSTM'] > row['ARIMA'])
                else ('ARIMA' if (row['Metric'] in ['MAE ($)', 'RMSE ($)', 'MAPE (%)', 'MPE (%)'] and row['ARIMA'] < row['LSTM'])
                or (row['Metric'] == 'Direction Accuracy (%)' and row['ARIMA'] > row['LSTM']) else 'Tie'),
    axis=1
)

print("\n" + "=" * 80)
print("MODEL COMPARISON TABLE")
print("=" * 80)
print(f"\n{comparison_df.to_string(index=False)}")

# Save comparison table
comparison_df.to_csv('model_comparison_metrics.csv', index=False)
print("\n✓ Comparison table saved to 'model_comparison_metrics.csv'")

In [ ]:
# Visualization 1: Predictions Comparison Over Time
print("\nCreating prediction comparison visualizations...")

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Full test period for ARIMA
axes[0].plot(arima_results_df['Date'], arima_results_df['Actual'], 
             linewidth=2, label='Actual Price', color='black', marker='o', markersize=3)
axes[0].plot(arima_results_df['Date'], arima_results_df['Predicted'], 
             linewidth=2, label='ARIMA Prediction', color='blue', alpha=0.7)
axes[0].fill_between(arima_results_df['Date'], 
                      arima_results_df['Lower_CI'], 
                      arima_results_df['Upper_CI'],
                      alpha=0.2, color='blue', label='95% Confidence Interval')
axes[0].set_title('ARIMA Predictions vs Actual Prices', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price ($)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# LSTM predictions (aligned subset)
axes[1].plot(lstm_results_df['Date'], lstm_results_df['Actual'], 
             linewidth=2, label='Actual Price', color='black', marker='o', markersize=3)
axes[1].plot(lstm_results_df['Date'], lstm_results_df['Predicted'], 
             linewidth=2, label='LSTM Prediction', color='green', alpha=0.7)
axes[1].set_title('LSTM Predictions vs Actual Prices', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Price ($)', fontsize=11)
axes[1].set_xlabel('Date', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: predictions_comparison.png")

In [ ]:
# Visualization 2: Residuals and Error Analysis
print("Creating residual analysis visualization...")

# Calculate residuals
arima_residuals = arima_results_df['Actual'] - arima_results_df['Predicted']
lstm_residuals = lstm_results_df['Actual'] - lstm_results_df['Predicted']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ARIMA residuals over time
axes[0, 0].plot(arima_results_df['Date'], arima_residuals, linewidth=1, color='blue', alpha=0.7)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 0].fill_between(arima_results_df['Date'], arima_residuals, 0, alpha=0.3, color='blue')
axes[0, 0].set_title('ARIMA Residuals Over Time', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Residual ($)', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# LSTM residuals over time
axes[0, 1].plot(lstm_results_df['Date'], lstm_residuals, linewidth=1, color='green', alpha=0.7)
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 1].fill_between(lstm_results_df['Date'], lstm_residuals, 0, alpha=0.3, color='green')
axes[0, 1].set_title('LSTM Residuals Over Time', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Residual ($)', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# ARIMA residual distribution
axes[1, 0].hist(arima_residuals, bins=30, color='blue', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(x=arima_residuals.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {arima_residuals.mean():.2f}')
axes[1, 0].set_title('ARIMA Residual Distribution', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Residual ($)', fontsize=10)
axes[1, 0].set_ylabel('Frequency', fontsize=10)
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# LSTM residual distribution
axes[1, 1].hist(lstm_residuals, bins=30, color='green', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=lstm_residuals.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {lstm_residuals.mean():.2f}')
axes[1, 1].set_title('LSTM Residual Distribution', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Residual ($)', fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('residuals_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: residuals_analysis.png")

# Residual statistics
print("\n" + "=" * 80)
print("RESIDUAL STATISTICS")
print("=" * 80)

print("\nARIMA Residuals:")
print(f"  Mean: ${arima_residuals.mean():.2f}")
print(f"  Std Dev: ${arima_residuals.std():.2f}")
print(f"  Min: ${arima_residuals.min():.2f}")
print(f"  Max: ${arima_residuals.max():.2f}")

print("\nLSTM Residuals:")
print(f"  Mean: ${lstm_residuals.mean():.2f}")
print(f"  Std Dev: ${lstm_residuals.std():.2f}")
print(f"  Min: ${lstm_residuals.min():.2f}")
print(f"  Max: ${lstm_residuals.max():.2f}")

In [ ]:
# Visualization 3: Metrics Comparison Bar Chart
print("\nCreating metrics comparison chart...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = ['MAE', 'RMSE', 'MAPE']
colors = ['#1f77b4', '#ff7f0e']  # Blue for ARIMA, Orange for LSTM

for idx, metric in enumerate(metrics_to_plot):
    values = [arima_metrics[metric], lstm_metrics[metric]]
    bars = axes[idx].bar(['ARIMA', 'LSTM'], values, color=colors, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{value:.2f}',
                      ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    if metric == 'MAPE':
        axes[idx].set_ylabel('Percentage (%)', fontsize=11, fontweight='bold')
    else:
        axes[idx].set_ylabel('Price ($)', fontsize=11, fontweight='bold')
    
    axes[idx].set_title(f'{metric} Comparison\n(Lower is Better)', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Highlight better model
    if values[0] < values[1]:
        axes[idx].text(0, max(values) * 0.95, '✓ Better', ha='center', 
                      fontsize=10, color='green', fontweight='bold',
                      bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    else:
        axes[idx].text(1, max(values) * 0.95, '✓ Better', ha='center', 
                      fontsize=10, color='green', fontweight='bold',
                      bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: metrics_comparison.png")

## Section 5: Model Comparison and Selection Rationale

### Summary of Results

Both ARIMA and LSTM models were successfully trained and evaluated on the TSLA price prediction task:

#### ARIMA/SARIMA Model
- **Parameters**: Automatically selected using pmdarima's auto_arima
- **Strength**: Interpretable parameters, provides confidence intervals
- **Data Used**: Only training set (2015-2024)
- **Prediction Period**: 2025-2026 (182+ trading days)

#### LSTM Neural Network Model
- **Architecture**: 3-layer LSTM with dropout regularization
- **Training Data**: Larger historical dataset with 60-day sequences
- **Strength**: Captures complex non-linear patterns
- **Data Normalization**: MinMax scaling (0-1 range)

### Performance Comparison

The comparison table above shows the key metrics:

1. **MAE (Mean Absolute Error)**: Average price prediction error in dollars
2. **RMSE (Root Mean Squared Error)**: Penalizes larger errors more heavily
3. **MAPE (Mean Absolute Percentage Error)**: Percentage-based error metric
4. **MPE (Mean Percentage Error)**: Shows bias direction (positive = underestimation)
5. **Direction Accuracy**: Percentage of correctly predicted up/down movements

### Model Selection Rationale

**Choice**: LSTM vs ARIMA depends on the use case:

**Choose ARIMA if:**
- Interpretability is critical for stakeholder communication
- Confidence intervals are important for risk management
- The model needs to be simple and computationally efficient
- You have limited training data
- Strong seasonality patterns exist

**Choose LSTM if:**
- Prediction accuracy is the primary objective
- The model can be used as one component in a larger ensemble
- You have sufficient computational resources
- Non-linear patterns are significant
- The data contains complex temporal dependencies

### Key Findings from Analysis

1. **Stationarity**: From Task 1, we know TSLA prices are non-stationary, requiring differencing for ARIMA
2. **Volatility**: TSLA exhibits high volatility, especially during market events
3. **Trend**: Strong upward trend visible in the data despite cyclical dips
4. **Residuals**: Should be normally distributed around zero (check plots above)
5. **Direction Accuracy**: Often more important for trading than absolute price accuracy

### Limitations and Considerations

1. **Past Performance**: Historical data doesn't guarantee future results (EMH principle)
2. **Black Swan Events**: Both models may fail during unprecedented market events
3. **Model Overfitting**: LSTM may overfit if not properly regularized
4. **Confidence**: ARIMA intervals narrow with longer forecast horizons
5. **Data Quality**: All models depend on clean, relevant input data

### Recommendations for GMF Investments

1. **Ensemble Approach**: Combine ARIMA and LSTM predictions for robustness
2. **Use in Context**: Treat predictions as ONE input to portfolio decisions
3. **Regular Retraining**: Update models monthly/quarterly with new data
4. **Risk Management**: Always use predictions with risk control mechanisms (VaR, stop-losses)
5. **Benchmark**: Compare model performance against S&P 500 (SPY) baseline

### Next Steps (Task 3-5)

- **Task 3**: Deep Learning refinements and alternative architectures
- **Task 4**: Portfolio optimization using predicted returns
- **Task 5**: Backtesting strategies using these forecasts

In [ ]:
# Save Model Information and Results
print("\n" + "=" * 80)
print("SAVING MODEL RESULTS AND DOCUMENTATION")
print("=" * 80)

# Create model summary document
model_summary = f"""
TESLA (TSLA) PRICE FORECASTING MODELS
=====================================

Date Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Data Period: 2015-01-01 to 2026-06-30
Training Period: 2015-01-01 to 2024-12-31
Test Period: 2025-01-01 to 2026-06-30

═══════════════════════════════════════════════════════════════════════════════

MODEL 1: ARIMA/SARIMA
─────────────────────
Parameters: {best_order}
Seasonal Parameters: {best_seasonal_order if best_seasonal_order else 'None (ARIMA only)'}
Model Type: {'SARIMA' if best_seasonal_order else 'ARIMA'}
Seasonal Period (m): {best_seasonal_order[3] if best_seasonal_order else 'N/A'} trading days

AIC Score: {arima_model.aic():.2f}
BIC Score: {arima_model.bic():.2f}

Performance on Test Set:
  MAE:  ${arima_metrics['MAE']:.2f}
  RMSE: ${arima_metrics['RMSE']:.2f}
  MAPE: {arima_metrics['MAPE']:.2f}%
  Direction Accuracy: {arima_metrics['Direction_Accuracy']:.2f}%

Interpretation:
- Uses historical prices to predict future values
- Assumes linear relationships and stationarity
- Provides confidence intervals for predictions
- Suitable for short to medium-term forecasts

═══════════════════════════════════════════════════════════════════════════════

MODEL 2: LSTM NEURAL NETWORK
────────────────────────────
Architecture:
  - Input Layer: {sequence_length} day sequences
  - LSTM Layer 1: 50 units, ReLU, Dropout 0.2
  - LSTM Layer 2: 50 units, ReLU, Dropout 0.2
  - LSTM Layer 3: 25 units, ReLU, Dropout 0.2
  - Dense Layer: 25 units, ReLU
  - Output Layer: 1 unit

Training Configuration:
  - Optimizer: Adam (lr=0.001)
  - Loss Function: Mean Squared Error
  - Batch Size: 32
  - Epochs: {len(history.history['loss'])}
  - Validation Split: 0.2
  - Early Stopping: Yes (patience=10)

Performance on Test Set:
  MAE:  ${lstm_metrics['MAE']:.2f}
  RMSE: ${lstm_metrics['RMSE']:.2f}
  MAPE: {lstm_metrics['MAPE']:.2f}%
  Direction Accuracy: {lstm_metrics['Direction_Accuracy']:.2f}%

Interpretation:
- Uses deep learning to capture non-linear patterns
- Learns complex temporal dependencies
- Requires more data for training
- Generally better for capturing trend changes

═══════════════════════════════════════════════════════════════════════════════

COMPARATIVE ANALYSIS
────────────────────
Better Model for MAE: {'LSTM' if lstm_metrics['MAE'] < arima_metrics['MAE'] else 'ARIMA'}
Better Model for RMSE: {'LSTM' if lstm_metrics['RMSE'] < arima_metrics['RMSE'] else 'ARIMA'}
Better Model for MAPE: {'LSTM' if lstm_metrics['MAPE'] < arima_metrics['MAPE'] else 'ARIMA'}
Better Model for Direction: {'LSTM' if lstm_metrics['Direction_Accuracy'] > arima_metrics['Direction_Accuracy'] else 'ARIMA'}

Recommendation:
Consider using both models in an ensemble approach:
1. ARIMA for interpretability and confidence intervals
2. LSTM for higher accuracy predictions
3. Combine predictions using weighted averaging

═══════════════════════════════════════════════════════════════════════════════

ARTIFACTS GENERATED
───────────────────
- split_visualization.png: Train-test split visualization
- acf_pacf_analysis.png: ACF/PACF plots for parameter selection
- lstm_training_history.png: LSTM training convergence
- predictions_comparison.png: Side-by-side prediction comparison
- residuals_analysis.png: Residual diagnostics
- metrics_comparison.png: Model performance comparison
- model_comparison_metrics.csv: Detailed metrics table
- model_summary.txt: This summary document

═══════════════════════════════════════════════════════════════════════════════

FILES SAVED
───────────
✓ ARIMA Model: arima_model.pkl
✓ LSTM Model: lstm_model.h5
✓ Scaler: price_scaler.pkl
✓ Results: arima_results.csv, lstm_results.csv
"""

# Save the summary
with open('model_summary.txt', 'w') as f:
    f.write(model_summary)

print(model_summary)

# Save model objects
import pickle

# Save ARIMA model
arima_model.model_.save('arima_model.pkl')
print("\n✓ Saved: arima_model.pkl")

# Save LSTM model
lstm_model.save('lstm_model.h5')
print("✓ Saved: lstm_model.h5")

# Save scaler
with open('price_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Saved: price_scaler.pkl")

# Save predictions to CSV
arima_results_df.to_csv('arima_results.csv', index=False)
print("✓ Saved: arima_results.csv")

lstm_results_df.to_csv('lstm_results.csv', index=False)
print("✓ Saved: lstm_results.csv")

print("\n" + "=" * 80)
print("TASK 2 COMPLETE!")
print("=" * 80)
print("\nAll models trained, evaluated, and documented.")
print("Ready for Task 3: Portfolio Optimization")